<a href="https://colab.research.google.com/github/HvdM17/web_car_price_estimator/blob/feat%2Farin-ml/training/02_modelling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

# Mengambil dataset bersih dari GitHub
url = "https://raw.githubusercontent.com/HvdM17/web_car_price_estimator/feat/arin-ml/data/cardekho_clean.csv"

df = pd.read_csv(url)

# Menampilkan ukuran dan 5 data pertama
print("Ukuran dataset:", df.shape)
display(df.head())

Ukuran dataset: (1133, 14)


,car_name,registration_year,insurance_validity,fuel_type,seats,kms_driven,ownsership,transmission,manufacturing_year,mileage(kmpl),engine(cc),max_power(bhp),torque(Nm),price(in lakhs)
0,2017 Mercedes-Benz S-Class S400,Jul-17,Comprehensive,Petrol,5,56000,First Owner,Automatic,2017,7.81,2996.0,2996.0,333.0,63.75
1,2020 Nissan Magnite Turbo CVT XV Premium Opt BSVI,Jan-21,Comprehensive,Petrol,5,30615,First Owner,Automatic,2020,17.40,999.0,999.0,9863.0,8.99
2,2018 BMW X1 sDrive 20d xLine,Sep-18,Comprehensive,Diesel,5,24000,First Owner,Automatic,2018,20.68,1995.0,1995.0,188.0,23.75
3,2019 Kia Seltos GTX Plus,Dec-19,Comprehensive,Petrol,5,18378,First Owner,Manual,2019,16.50,1353.0,1353.0,13808.0,13.56
4,2019 Skoda Superb LK 1.8 TSI AT,Aug-19,Comprehensive,Petrol,5,44900,First Owner,Automatic,2019,14.67,1798.0,1798.0,17746.0,24.00


In [2]:
# Melihat informasi tipe data
df.info()

# Memeriksa missing value dan duplikasi
print("\nMissing value:")
print(df.isnull().sum())

print("\nJumlah duplikasi:", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1133 entries, 0 to 1132
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   car_name            1133 non-null   object 
 1   registration_year   1133 non-null   object 
 2   insurance_validity  1133 non-null   object 
 3   fuel_type           1133 non-null   object 
 4   seats               1133 non-null   int64  
 5   kms_driven          1133 non-null   int64  
 6   ownsership          1133 non-null   object 
 7   transmission        1133 non-null   object 
 8   manufacturing_year  1133 non-null   object 
 9   mileage(kmpl)       1133 non-null   float64
 10  engine(cc)          1133 non-null   float64
 11  max_power(bhp)      1133 non-null   float64
 12  torque(Nm)          1133 non-null   float64
 13  price(in lakhs)     1133 non-null   float64
dtypes: float64(5), int64(2), object(7)
memory usage: 124.1+ KB

Missing value:
car_name              0
regis

In [3]:
# Mengelompokkan kolom berdasarkan tipe data
kolom_numerik = df.select_dtypes(include=np.number).columns.tolist()
kolom_kategori = df.select_dtypes(include='object').columns.tolist()

print("Kolom numerik:", kolom_numerik)
print("Kolom kategorikal:", kolom_kategori)

Kolom numerik: ['seats', 'kms_driven', 'mileage(kmpl)', 'engine(cc)', 'max_power(bhp)', 'torque(Nm)', 'price(in lakhs)']
Kolom kategorikal: ['car_name', 'registration_year', 'insurance_validity', 'fuel_type', 'ownsership', 'transmission', 'manufacturing_year']


In [4]:
# Memeriksa outlier pada kolom numerik
for kolom in kolom_numerik:
    Q1 = df[kolom].quantile(0.25)
    Q3 = df[kolom].quantile(0.75)
    IQR = Q3 - Q1

    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR

    jumlah = ((df[kolom] < batas_bawah) |
              (df[kolom] > batas_atas)).sum()

    print(f"{kolom}: {jumlah} outlier")

seats: 137 outlier
kms_driven: 14 outlier
mileage(kmpl): 165 outlier
engine(cc): 95 outlier
max_power(bhp): 95 outlier
torque(Nm): 60 outlier
price(in lakhs): 147 outlier


In [5]:
# Menentukan kategori yang valid
valid_fuel = ['Petrol', 'Diesel', 'CNG']
valid_ownership = [
    'First Owner', 'Second Owner', 'Third Owner',
    'Fourth Owner', 'Fifth Owner'
]
valid_transmission = ['Manual', 'Automatic']

# Menghapus data dengan kategori tidak valid dan harga ekstrem
df_model = df[
    (df['price(in lakhs)'] <= 100) &
    (df['fuel_type'].isin(valid_fuel)) &
    (df['ownsership'].isin(valid_ownership)) &
    (df['transmission'].isin(valid_transmission))
].copy()

print("Ukuran data setelah cleaning:", df_model.shape)
print("Total missing value:", df_model.isnull().sum().sum())

Ukuran data setelah cleaning: (1106, 14)
Total missing value: 0


In [6]:
# Menentukan kolom fitur input
fitur = [
    'insurance_validity',
    'fuel_type',
    'seats',
    'kms_driven',
    'ownsership',
    'transmission',
    'manufacturing_year'
]

# Memisahkan fitur dan target
X = df_model[fitur].copy()
y = df_model['price(in lakhs)'].copy()

# Menampilkan ukuran data
print("Ukuran fitur X:", X.shape)
print("Ukuran target y:", y.shape)

display(X.head())

Ukuran fitur X: (1106, 7)
Ukuran target y: (1106,)


,insurance_validity,fuel_type,seats,kms_driven,ownsership,transmission,manufacturing_year
0,Comprehensive,Petrol,5,56000,First Owner,Automatic,2017
1,Comprehensive,Petrol,5,30615,First Owner,Automatic,2020
2,Comprehensive,Diesel,5,24000,First Owner,Automatic,2018
3,Comprehensive,Petrol,5,18378,First Owner,Manual,2019
4,Comprehensive,Petrol,5,44900,First Owner,Automatic,2019


In [7]:
# Menentukan fitur kategorikal
categorical_features = [
    'insurance_validity',
    'fuel_type',
    'ownsership',
    'transmission'
]

# Menentukan fitur numerik
numeric_features = [
    'seats',
    'kms_driven',
    'manufacturing_year'
]

print("Fitur kategorikal:", categorical_features)
print("Fitur numerik:", numeric_features)

Fitur kategorikal: ['insurance_validity', 'fuel_type', 'ownsership', 'transmission']
Fitur numerik: ['seats', 'kms_driven', 'manufacturing_year']


In [8]:
from sklearn.model_selection import train_test_split

# Membagi data menjadi training dan testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Menampilkan ukuran masing-masing data
print("Data training:", X_train.shape)
print("Data testing:", X_test.shape)
print("Target training:", y_train.shape)
print("Target testing:", y_test.shape)

Data training: (884, 7)
Data testing: (222, 7)
Target training: (884,)
Target testing: (222,)


In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Membuat preprocessing untuk setiap jenis fitur
preprocessor = ColumnTransformer(
    transformers=[
        ('kategori', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('numerik', StandardScaler(), numeric_features)
    ]
)

print("Preprocessing berhasil dibuat!")

Preprocessing berhasil dibuat!


In [10]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor

# Menyiapkan algoritma yang akan dibandingkan
models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(
        n_estimators=100,
        random_state=42
    )
}

# Menggabungkan preprocessing dan algoritma
pipelines = {
    nama: Pipeline([
        ('preprocessor', preprocessor),
        ('model', algoritma)
    ])
    for nama, algoritma in models.items()
}

print("3 Pipeline model berhasil dibuat!")

3 Pipeline model berhasil dibuat!


In [11]:
from sklearn.model_selection import KFold, cross_validate

# Mengatur 5-Fold Cross Validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)

# Menentukan metrik evaluasi
scoring = {
    'MAE': 'neg_mean_absolute_error',
    'RMSE': 'neg_root_mean_squared_error',
    'R2': 'r2'
}

# Menguji seluruh model menggunakan Cross Validation
hasil_cv = []

for nama, pipeline in pipelines.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring
    )

    hasil_cv.append({
        'Model': nama,
        'MAE': -scores['test_MAE'].mean(),
        'RMSE': -scores['test_RMSE'].mean(),
        'R2': scores['test_R2'].mean()
    })

# Menampilkan hasil perbandingan
hasil_cv = pd.DataFrame(hasil_cv)
display(hasil_cv)

,Model,MAE,RMSE,R2
0,Linear Regression,8.899236,14.400439,0.326988
1,Decision Tree,9.777349,19.135816,-0.194077
2,Random Forest,7.656903,14.192783,0.348222


In [12]:
from sklearn.model_selection import GridSearchCV

# Menentukan kombinasi parameter Random Forest
param_grid = {
    'model__n_estimators': [100, 200],
    'model__max_depth': [None, 10, 20],
    'model__min_samples_split': [2, 5]
}

# Membuat GridSearchCV
grid_search = GridSearchCV(
    estimator=pipelines['Random Forest'],
    param_grid=param_grid,
    cv=cv,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

# Melakukan pencarian parameter terbaik
grid_search.fit(X_train, y_train)

print("Parameter terbaik:", grid_search.best_params_)
print("MAE Cross Validation terbaik:", -grid_search.best_score_)

Fitting 5 folds for each of 12 candidates, totalling 60 fits
Parameter terbaik: {'model__max_depth': None, 'model__min_samples_split': 5, 'model__n_estimators': 200}
MAE Cross Validation terbaik: 7.533963963490507


In [13]:
#evaluasi model pada data testing

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# Mengambil pipeline dengan parameter terbaik
best_pipeline = grid_search.best_estimator_

# Melakukan prediksi pada data testing
y_pred = best_pipeline.predict(X_test)

# Menghitung evaluasi model
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Hasil Evaluasi Random Forest:")
print("MAE :", mae)
print("RMSE:", rmse)
print("R2  :", r2)

Hasil Evaluasi Random Forest:
MAE : 6.00613969520157
RMSE: 11.600432455587574
R2  : 0.3699184890661885


In [14]:
import joblib
import os

# Membuat folder models jika belum tersedia
os.makedirs('models', exist_ok=True)

# Menyimpan model terbaik
joblib.dump(best_pipeline, 'models/car_price_model.joblib')

# Mengecek file model
print("Model berhasil disimpan!")
print("Ukuran file:", os.path.getsize('models/car_price_model.joblib'), "bytes")

Model berhasil disimpan!
Ukuran file: 6711178 bytes


In [17]:
from google.colab import files

files.download('models/car_price_model.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>